# Polygonal macro meshes

Local triangulation preserves the original macro faces. The research campaign uses the oscillatory PDE and P1/P3 degrees of the 2024 generalized RAD paper on five families: triangles, squares, rhombi, L-shaped cells and hexagons. Boundary-separated centroid fans satisfy the distinct-adjacent-triangle condition.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/60f6b4e70c2acaefdb1b76dd131ba4de880fd3a980137799490357f324e6e6ab/30_polygonal_macro_meshes-companion.zip"
COMPANION_SHA256 = "60f6b4e70c2acaefdb1b76dd131ba4de880fd3a980137799490357f324e6e6ab"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/30_polygonal_macro_meshes.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Declare the physical variational forms

The application defines local operators with the shared element, integration
and oriented face kernels and supplies the actual A/B/C blocks to the generic
assembler. For primal pressure the form is (K grad(p),grad(v)); mixed coordinates
use the inverse-permeability flux mass, divergence and normal-moment constraint.

$$
\begin{aligned}
A_Tu_T+B_T\lambda&=f_T,\\
C_Tu_T+D_T\lambda&=g_T, & C_T&=-B_T^{\mathsf T}, & D_T&=0.
\end{aligned}
$$

The local constant pressure and its physical integral moment retain the executed
nodal/modal basis. The separate global `Equation` supplies the signed pressure
boundary functional; pure Neumann data add a physical pressure integral constraint.
The flux trace follows fixed macroface normals. Mixed flux coordinates retain
their H(div) normal/interior moments and Piola transform; a primal gradient
field is evaluated independently. Generic assembly/solve and physical coefficient
interpretation are separate operations.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.darcy import define_darcy, recover_darcy

import numpy as np
from pymhm import PolygonMesh

mesh = PolygonMesh(
    np.array([[0, 0], [1, 0], [1, 0.5], [0.5, 0.5], [0.5, 1], [0, 1], [1, 1]]),
    (np.array([0, 1, 2, 3, 4, 5]), np.array([3, 2, 6, 4])),
    local_triangulation="boundary",
)
exact = lambda x: 1 + x[:, 0] + 2 * x[:, 1]
solution_definition = define_darcy(mesh, degree=2, dirichlet=exact, local_refinement=2)
solution_system = assemble(solution_definition.problem)
solution = recover_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
assert solution.l2_error(exact) < 1e-11
solution.l2_error(exact)


The full research campaign is acquired separately from this lightweight notebook. The following cells display its numerical record and publication figures.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    from pathlib import Path
    import json
    from IPython.display import Image, display
    record = json.loads((root / "examples/results/polygons.json").read_text())
    record


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/polygons/convergence.png")))
    display(Image(filename=str(root / "docs/figures/polygons/fields.png")))
